# Evaluation — Baseline vs Augmented Model Comparison
**DiagGen+ | Advanced AI Group Project 2026**

Run this notebook **after both Phase 1 and Phase 2 are complete**.
It loads the saved metric files and produces all figures needed for the report.

No model inference here — all computation is already done and saved.

In [ ]:
import sys
sys.path.insert(0, '../..')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

FIGURES_DIR   = Path('../../reports/figures')
PROCESSED_DIR = Path('../../data/processed')

baseline  = json.loads((PROCESSED_DIR / 'baseline_metrics.json').read_text())
augmented = json.loads((PROCESSED_DIR / 'augmented_metrics.json').read_text())
rare_meta = json.loads((PROCESSED_DIR / 'rare_classes.json').read_text())
RARE_DISEASES = rare_meta['rare_diseases']
print('Metrics loaded.')

## 1. Overall Metrics Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Accuracy','Macro F1','Weighted F1','Rare class mean F1','Common class mean F1'],
    'Baseline': [
        baseline['accuracy'], baseline['macro_f1'], baseline['weighted_f1'],
        baseline['rare_classes_mean_f1'], baseline['common_classes_mean_f1']
    ],
    'Augmented': [
        augmented['accuracy'], augmented['macro_f1'], augmented['weighted_f1'],
        augmented['rare_classes_mean_f1'], augmented['common_classes_mean_f1']
    ],
})
summary['Delta']      = summary['Augmented'] - summary['Baseline']
summary['Delta (%)']  = (summary['Delta'] / summary['Baseline'] * 100).round(1)
summary = summary.round(4)
print(summary.to_string(index=False))
summary.to_csv(FIGURES_DIR / 'summary_table.csv', index=False)
print('\nSaved to reports/figures/summary_table.csv')

## 2. Per-Class F1 Comparison Chart

In [ ]:
label_names = list(baseline['per_class_f1'].keys())
b_f1 = [baseline['per_class_f1'].get(n, 0) for n in label_names]
a_f1 = [augmented['per_class_f1'].get(n, 0) for n in label_names]

x   = np.arange(len(label_names))
fig, ax = plt.subplots(figsize=(16, 6))
bars_b = ax.bar(x - 0.2, b_f1, 0.38, label='Baseline',  color='#5B9BD5', alpha=0.85)
bars_a = ax.bar(x + 0.2, a_f1, 0.38, label='Augmented', color='#ED7D31', alpha=0.85)

# Highlight rare class pairs
for i, name in enumerate(label_names):
    if name in RARE_DISEASES:
        ax.axvspan(i - 0.45, i + 0.45, alpha=0.08, color='red')

ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1: Baseline vs VAE-Augmented Model\n(Red shading = simulated rare classes)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'f1_comparison_final.png', dpi=150)
plt.show()
print('Figure saved.')

## 3. Rare vs Common Class F1 — Grouped Bar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
categories = ['Common classes', 'Rare classes']
base_vals  = [baseline['common_classes_mean_f1'],  baseline['rare_classes_mean_f1']]
aug_vals   = [augmented['common_classes_mean_f1'], augmented['rare_classes_mean_f1']]
x = np.arange(2)
ax.bar(x - 0.2, base_vals, 0.38, label='Baseline',  color='#5B9BD5')
ax.bar(x + 0.2, aug_vals,  0.38, label='Augmented', color='#ED7D31')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylabel('Mean F1 Score', fontsize=12)
ax.set_title('VAE Augmentation Impact:\nCommon vs Rare Class F1', fontsize=12)
ax.set_ylim(0, 1.1)
ax.legend()
for bar in ax.patches:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rare_vs_common_f1.png', dpi=150)
plt.show()
print('Figure saved.')

## Figures Checklist for Report

| Figure file | Report section | Caption |
|-------------|---------------|--------|
| `class_distribution.png` | Methodology | Original balanced dataset class distribution |
| `class_imbalance_simulation.png` | Methodology | Controlled imbalance simulation (Phase 0) |
| `baseline_confusion_matrix.png` | Results | Baseline BERT confusion matrix |
| `baseline_per_class_f1.png` | Results | Baseline per-class F1 (red = rare classes) |
| `f1_comparison_final.png` | Results | Baseline vs Augmented per-class F1 |
| `rare_vs_common_f1.png` | Results | VAE augmentation impact on rare vs common classes |